# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We will display a list of record sets (with their `@id`), and for each, the fields and columns with their `@id` and name.

In [ ]:
record_sets = list(dataset.record_sets())
print("Record Sets:")
for rs in record_sets:
    print(f"- RecordSet @id: {rs['@id']}, name: {rs.get('name', 'N/A')}")
    fields = rs.get('fields', [])
    if fields:
        print("  Fields:")
        for f in fields:
            print(f"    - Field @id: {f['@id']} name: {f.get('name', 'N/A')}, dataType: {f.get('dataType', 'N/A')}")
        # Show columns if present
        for f in fields:
            columns = f.get('columns', [])
            if columns:
                print("      Columns:")
                for c in columns:
                    print(f"        * Column @id: {c['@id']} name: {c.get('name', 'N/A')}, dataType: {c.get('dataType', 'N/A')}")
    else:
        print("  No fields found.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

Below, the available record sets (referenced by their `@id`) are loaded into Pandas DataFrames.
For demonstration, we'll use the first record set found.

In [ ]:
dataframes = {}
record_set_ids = [rs['@id'] for rs in record_sets]
print('Loading record sets:', record_set_ids)

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records for RecordSet {record_set_id}")

# Display columns for the first record set
first_record_set_id = record_set_ids[0] if record_set_ids else None
if first_record_set_id:
    print(f"Columns in record set {first_record_set_id}:\n{dataframes[first_record_set_id].columns.tolist()}")
    dataframes[first_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We will:
- Select a numeric field (e.g., 'age') referenced by its `@id`,
- Filter records for a threshold,
- Normalize the selected numeric field,
- Group by a categorical field referenced by its `@id` (e.g., 'sex').

Update `numeric_field_id` and `group_field_id` below to match the actual field `@id`s from dataset overview.

In [ ]:
# Example field @ids for demonstration:
# Please update with field IDs from the above overview for your dataset
numeric_field_id = None
group_field_id = None

# Search for a numeric field (Integer or Float), e.g. Age
if first_record_set_id:
    for rs in record_sets:
        if rs['@id'] == first_record_set_id:
            fields = rs.get('fields', [])
            for f in fields:
                # Identify numeric fields
                if f.get('dataType') in ['schema:Integer', 'schema:Float'] or 'age' in f.get('name', '').lower():
                    numeric_field_id = f['@id']
                if f.get('dataType') == 'schema:Boolean' or 'sex' in f.get('name', '').lower():
                    group_field_id = f['@id']
            break

if not numeric_field_id:
    # fallback to column names
    possible_numeric = [col for col in dataframes[first_record_set_id].columns if 'age' in col.lower()]
    if possible_numeric:
        numeric_field_id = possible_numeric[0]

if not group_field_id:
    possible_group = [col for col in dataframes[first_record_set_id].columns if 'sex' in col.lower()]
    if possible_group:
        group_field_id = possible_group[0]

print(f'Numeric field: {numeric_field_id}')
print(f'Group/categorical field: {group_field_id}')

# Proceed with EDA
threshold = 50  # age threshold example
df = dataframes[first_record_set_id]
if numeric_field_id and numeric_field_id in df.columns:
    # Ensure numeric field is numeric
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"Grouped data by {group_field_id} (showing mean of {numeric_field_id}):")
        print(grouped_df.head())
else:
    print("No appropriate numeric field found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Here, we will visualize the distribution of the numeric field (e.g., age) and compare group means (e.g., by sex), using matplotlib.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram for the numeric field
if numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=10, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # Grouped boxplot
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion
This notebook demonstrated loading, exploring, and visualizing the FAIR^2 dataset using the `mlcroissant` library.

- Dataset metadata is accessible via the Croissant schema.
- Records were loaded and filtered by record set `@id`.
- Numeric fields (such as age) were analyzed and visualized.
- All entities (record sets, fields) referenced by their `@id` for reproducibility.

Further analysis can be performed by referencing specific fields and applying domain-specific methods as needed.